In [ ]:
import os
import sys
import argparse
from pathlib import Path
import time

import torch
import torch.nn as nn
from torch import Tensor
from torch.optim import AdamW, Optimizer
from torch.utils.data import DataLoader, random_split, Subset

In [ ]:
cwd = Path.cwd()
print("Current working directory:", cwd)
root_path = Path("/path/to/BrainWear_Kareem")
project_root = root_path / "FYP"
print("Project root:", project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from datasets.brats2020_png import BraTS2020PNGDataset
from slot_attention.training_2d.slot_attention_2d import SlotClassifier2D

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f"Using device: {torch.cuda.get_device_name(0)}. Cuda available: True")
else:
    print(f"Using device: cpu. Cuda available: False")

exp_name = "v14a_0.15_test"
checkpoint_path = project_root/"slot_attention"/"training_2d"/"models"/"checkpoints"/f"brats_png_{exp_name}"/"ckpt.pt"
checkpoint = torch.load(checkpoint_path, map_location=device)

hp = checkpoint['hyperparameters']
model = SlotClassifier2D(
    in_shape=(hp['in_channels'], hp['input_h'], hp['input_w']),
    width=hp['width'],
    num_slots=hp['num_slots'],
    slot_dim=hp['slot_dim'],
    routing_iters=hp['routing_iters'],
    temperature=hp['temp'],
    encoder_depth=hp.get('encoder_depth', 4),
    enc3_init_skip=hp.get('enc3_init_skip', False),
    use_mask_pool_classifier=hp.get('use_mask_pool_classifier', False),
)

if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    model.load_state_dict(checkpoint)

model.to(device)
model.eval()
model.set_deterministic_slot_init(seed=42)

print(f"Model from {exp_name} loaded successfully.")
print(f"Hyperparameters: routing_iters={hp['routing_iters']}, num_slots={hp['num_slots']}, slot_dim={hp['slot_dim']}")
print(f"  encoder_depth={hp.get('encoder_depth', 4)}, enc3_init_skip={hp.get('enc3_init_skip', False)}")

In [ ]:
data_dir = root_path / "Processed_BraTS2020_TrainingData_PNG"
print("Data directory:", data_dir)

mini_dataset = BraTS2020PNGDataset(data_dir=str(data_dir), max_samples=3, shuffle=True)
# mini_dataset = BraTS2020PNGDataset(data_dir=str(data_dir), max_samples=3)
# mini_dataset = Subset(mini_dataset, [1])

# Load 3 samples
samples_data = []
for idx in range(3):
    t2_final, seg_final = mini_dataset[idx]
    test_mri = t2_final.unsqueeze(0).to(device)     # (1, 1, 240, 240)
    ground_truth_mask = seg_final.numpy()            # (240, 240)
    
    with torch.no_grad():
        recon_combined, recons, masks, slots, mlp_outputs = model(test_mri)
    
    samples_data.append({
        'test_mri': test_mri,
        'ground_truth_mask': ground_truth_mask,
        'recon_combined': recon_combined,
        'recons': recons,
        'masks': masks,
        'mlp_outputs': mlp_outputs
    })

print(f"Loaded {len(samples_data)} samples")
print(f"Input shape: {samples_data[0]['test_mri'].shape}")
print(f"Input device: {samples_data[0]['test_mri'].device}")

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# Extract data from first sample to get num_slots
num_slots = samples_data[0]['masks'].shape[1]

# BraTS colormap setup
brats_cmap = mcolors.ListedColormap(['black', 'red', 'yellow', 'blue'])
bounds = [-0.5, 0.5, 1.5, 2.5, 3.5]
norm = mcolors.BoundaryNorm(bounds, brats_cmap.N)

# Create 3×8 grid (3 scans × (original | original+GT | recon | 5 slots))
cols = num_slots + 2  # original | original+GT | recon | slot masks
rows = 3
fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
image_savedir = project_root / "slot_attention" / "training_2d" / "visualisations"

for row, sample_data in enumerate(samples_data):
    original_img = sample_data['test_mri'][0, 0].cpu().numpy()         # (240, 240)
    reconstructed = sample_data['recon_combined'][0, 0].cpu().numpy()   # (240, 240)
    slot_masks = sample_data['masks'][0, :, 0].cpu().numpy()            # (num_slots, 240, 240)
    ground_truth_mask = sample_data['ground_truth_mask']                 # (240, 240)
    
    gt_display = ground_truth_mask.copy()
    
    # Column 0: Original
    axes[row, 0].imshow(original_img, cmap='gray')
    axes[row, 0].set_title("Original" if row == 0 else "")
    axes[row, 0].axis('off')
    
    # Column 1: Original + GT overlay
    axes[row, 1].imshow(original_img, cmap='gray')
    gt_overlay = np.ma.masked_where(gt_display == 0, gt_display)
    axes[row, 1].imshow(gt_overlay, cmap=brats_cmap, norm=norm, alpha=0.45, interpolation='none')
    axes[row, 1].set_title("Original + GT" if row == 0 else "")
    axes[row, 1].axis('off')
    
    # Column 2: Reconstruction
    # axes[row, 2].imshow(reconstructed, cmap='gray')
    # axes[row, 2].set_title("Reconstruction" if row == 0 else "")
    # axes[row, 2].axis('off')
    
    # Columns 3+: Slot masks
    for i in range(num_slots):
        im = axes[row, i + 2].imshow(slot_masks[i], cmap='viridis', vmin=0, vmax=1)
        if row == 0:
            axes[row, i + 2].set_title(f"Slot {i+1}")
        axes[row, i + 2].axis('off')

# plt.colorbar(im, ax=axes[:, -1].tolist(), shrink=0.8, label='Mask weight')
# plt.suptitle("Slot Masks - BraTS", fontsize=14)
plt.tight_layout()
plt.savefig(image_savedir / f'{exp_name}.png', dpi=150, bbox_inches='tight')
print(f"Visualisation saved to {image_savedir / f'{exp_name}.png'}")
plt.show()

In [ ]:
from datasets.brainwear_png import BrainWearPNGDataset

bw_data_dir = root_path / "Processed_Brainwear_PNG_fixed_norm"
print("BrainWear data directory:", bw_data_dir)

bw_mini_dataset = BrainWearPNGDataset(root_dir=str(bw_data_dir), max_patients=3, shuffle=True)

# Load 3 patients – use the q50 (median) slice from each patient's 5-slice stack
bw_samples_data = []
for idx in range(3):
    t2_stack, _ = bw_mini_dataset[idx]                               # (5, H, W)
    t2_slice = t2_stack[2].unsqueeze(0).unsqueeze(0).to(device)      # q50 → (1, 1, H, W)

    with torch.no_grad():
        recon_combined, recons, masks, slots, mlp_outputs = model(t2_slice)

    bw_samples_data.append({
        'test_mri': t2_slice,
        'recon_combined': recon_combined,
        'recons': recons,
        'masks': masks,
        'mlp_outputs': mlp_outputs,
    })

print(f"Loaded {len(bw_samples_data)} BrainWear samples")
print(f"Input shape: {bw_samples_data[0]['test_mri'].shape}")

bw_num_slots = bw_samples_data[0]['masks'].shape[1]

cols = bw_num_slots + 1   # original | recon | slot masks
rows = 3
fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))

for row, sample_data in enumerate(bw_samples_data):
    original_img = sample_data['test_mri'][0, 0].cpu().numpy()         # (H, W)
    reconstructed = sample_data['recon_combined'][0, 0].cpu().numpy()   # (H, W)
    slot_masks = sample_data['masks'][0, :, 0].cpu().numpy()            # (num_slots, H, W)

    # Column 0: Original
    axes[row, 0].imshow(original_img, cmap='gray')
    axes[row, 0].set_title("Original" if row == 0 else "")
    axes[row, 0].axis('off')

    # Column 1: Reconstruction
    # axes[row, 1].imshow(reconstructed, cmap='gray')
    # axes[row, 1].set_title("Reconstruction" if row == 0 else "")
    # axes[row, 1].axis('off')

    # Columns 2+: Slot masks
    for i in range(bw_num_slots):
        im = axes[row, i + 1].imshow(slot_masks[i], cmap='viridis', vmin=0, vmax=1)
        if row == 0:
            axes[row, i + 1].set_title(f"Slot {i+1}")
        axes[row, i + 1].axis('off')

# plt.colorbar(im, ax=axes[:, -1].tolist(), shrink=0.8, label='Mask weight')
# plt.suptitle("Slot Masks - BrainWear", fontsize=14)
plt.tight_layout()
plt.savefig(image_savedir / f'{exp_name}_brainwear.png', dpi=150, bbox_inches='tight')
print(f"Visualisation saved to {image_savedir / f'{exp_name}_brainwear.png'}")
plt.show()
